# Cyberbullying Detection

Import necessary libraries and load dataset

In [132]:
import numpy as np
import pandas as pd
import string
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from nltk.tokenize import TweetTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FunctionTransformer, FeatureUnion
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score
from sklearn.svm import SVC

stop_words = set(stopwords.words("english"))
stemmer = SnowballStemmer(language="english")
tknzr = TweetTokenizer()

In [133]:
data = pd.read_csv("Formspring.csv")
data.head(4)

,text,answer
0,Q: what&#039;s your favorite song? :D<br>A: I ...,No
1,Q: <3<br>A: </3 ? haha jk! <33,No
2,Q: &quot;hey angel you duh sexy&quot;<br>A: R...,No
3,Q: (:<br>A: ;(,No


In [134]:
text_data = data["text"].to_numpy()
answer_data = data["answer"].map({"No": 0, "Yes": 1})

In [135]:
def process_text(input_text : str) -> str:
    input_text = input_text[3:].lower().replace("&#039;", '\'').replace("<br>a: ", " ", count=1)
    tokens = tknzr.tokenize(input_text)
    tokens = [word for word in tokens if (word not in stop_words and word not in string.punctuation)]

    for i in range(len(tokens)):
        tokens[i] = stemmer.stem(tokens[i])
    
    return " ".join(tokens)

In [136]:

def process_text_arr(text_data : np.ndarray) -> np.ndarray:
    td = text_data.copy()
    for i in range(text_data.shape[0]):
        td[i] = process_text(td[i])
    return td

In [137]:
X_train, X_test, y_train, y_test = train_test_split(text_data, answer_data, test_size=0.2, stratify=answer_data, random_state=42)


pipe = Pipeline([
    ("process", FunctionTransformer(process_text_arr)),
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.98,
        sublinear_tf=True,
        max_features=50_000
    )),
    ("model", SVC(
        kernel="linear",
        class_weight="balanced"
    ))
])

pipe.fit(X_train, y_train)

y_pred = pipe.predict(X_test)

In [138]:
def eval_model(y_true, y_pred):
    print(f"Accuracy score: {accuracy_score(y_true, y_pred)}")
    print(f"F1 score: {f1_score(y_true, y_pred)}")
    print(f"Precision score: {precision_score(y_true, y_pred)}")
    print(f"Recall score: {recall_score(y_true, y_pred)}")

In [140]:
eval_model(y_test, y_pred)

Accuracy score: 0.9372623574144486
F1 score: 0.5843828715365239
Precision score: 0.5110132158590308
Recall score: 0.6823529411764706
